In [2]:
import numpy as np
import torch
from torch import nn

### 1D - spinless

Let $\Lambda=\mathbb{Z}/L\mathbb{Z}$ be the 1D lattice with periodic boundary conditions.
Assume $N\le L$. Let $\psi:\Lambda^N\to\mathbb{F}$ ($\mathbb{F}=\mathbb{R}$ or $\mathbb{C}$)
be antisymmetric. Define the set of distinct-site configurations $\Omega:=\{(x_1,\dots,x_N)\in\Lambda^N:\ x_i\neq x_j \text{ in }\Lambda \text{ for all } i\neq j\}.$
Let $z_x:=e^{2\pi i x/L}\in\mathbb{C},\quad x\in\Lambda,$
and define
$F(x_1,\dots,x_N)
:=\prod_{1\le i<j\le N}\bigl(z_{x_j}-z_{x_i}\bigr).$
Then $F$ is antisymmetric and nonzero on $\Omega$, and there exists a symmetric function
$g:\Lambda^N\to\mathbb{F}$ such that
\begin{equation}
\psi(x_1,\dots,x_N)=F(x_1,\dots,x_N)g(x_1,\dots,x_N),
\qquad \forall (x_1,\dots,x_N)\in\Lambda^N.
\end{equation}

In [ ]:
class psi_1D_spinless(nn.Module):
    def __init__(self, L: int, dim_feedforward: int, activation: nn.Module = nn.GELU):
        super().__init__()
        self.L = L
        # weights, drawn from a Gaussian distribution (for now)
        self.w_s = nn.Parameter(torch.randn(L))
        # the MLP
        self.g_prime = nn.Sequential(
            nn.Linear(1, dim_feedforward),
            activation(),
            nn.Linear(dim_feedforward, dim_feedforward),
            activation(),
            nn.Linear(dim_feedforward, dim_feedforward),
            activation(),
            nn.Linear(dim_feedforward, dim_feedforward),
            activation(),
            nn.Linear(dim_feedforward, 2),
        )

    def F_antisymmetric(self, x: torch.Tensor):
        # convert to complex numbers z_x:=e^{2\pi i x/L}
        z_x = torch.exp((2 * torch.pi * 1j * x) / self.L)
        vandermonde_matrix = torch.vander(z_x, increasing=True)
        # take the determinant
        F = torch.linalg.det(vandermonde_matrix)
        return F

    def forward(self, x):
        # compute the antisymmetric function F
        F = self.F_antisymmetric(x)
        # compute the symmetric function g
        # first calculate eta
        s = torch.arange(1, self.L + 1).unsqueeze(-1)
        eta = torch.dot(self.w_s, torch.sum(s == x.unsqueeze(0), dim=-1).float())
        # now pass through the g' NN
        g = self.g_prime(eta.unsqueeze(0))
        g = torch.complex(g[..., 0], g[..., 1])
        # return the product of the two
        return F * g